In [10]:
import pandas as pd
import numpy as np
import pickle
import os

from sklearn.model_selection import train_test_split, KFold
from xgboost import XGBRegressor
from sklearn import metrics

# =========================
# PATH SETUP (IMPORTANT)
# =========================
DATA_PATH = "./Data/Datasetindia_house_dataset.csv"
MODEL_PATH = "./model"

os.makedirs(MODEL_PATH, exist_ok=True)

# =========================
# LOAD DATA
# =========================
df = pd.read_csv(datasets_path)
df.dropna(inplace=True)

# =========================
# TARGET LOG TRANSFORM
# =========================
df['rent'] = np.log1p(df['rent'])

# =========================
# TARGET ENCODING (K-FOLD SAFE)
# =========================
def target_encode(df, column, target):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    encoded = np.zeros(len(df))

    for train_idx, val_idx in kf.split(df):
        train_df = df.iloc[train_idx]
        val_df = df.iloc[val_idx]

        mean_map = train_df.groupby(column)[target].mean()
        encoded[val_idx] = val_df[column].map(mean_map)

    return encoded

df['locality_encoded'] = target_encode(df, 'locality', 'rent')
df['city_encoded'] = target_encode(df, 'city', 'rent')

# fill missing
df['locality_encoded'].fillna(df['rent'].mean(), inplace=True)
df['city_encoded'].fillna(df['rent'].mean(), inplace=True)

# =========================
# FURNISHING ENCODING
# =========================
furnishing_map = {
    'Unfurnished': 0,
    'Semi-Furnished': 1,
    'Furnished': 2
}
df['furnishing_encoded'] = df['furnishing'].map(furnishing_map)

# =========================
# FEATURE ENGINEERING
# =========================
df['area_per_bed'] = df['area'] / (df['beds'] + 1)
df['bath_per_bed'] = df['bathrooms'] / (df['beds'] + 1)
df['total_rooms'] = df['beds'] + df['bathrooms']
df['beds_area'] = df['beds'] * df['area']

# =========================
# FEATURES
# =========================
X = df[[
    'area',
    'beds',
    'bathrooms',
    'balconies',
    'area_per_bed',
    'bath_per_bed',
    'total_rooms',
    'beds_area',
    'locality_encoded',
    'city_encoded',
    'furnishing_encoded'
]]

y = df['rent']

# =========================
# TRAIN TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# MODEL TRAINING
# =========================
model = XGBRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

# =========================
# EVALUATION
# =========================
y_pred = model.predict(X_test)

print("R2 Score:", metrics.r2_score(y_test, y_pred))

y_test_actual = np.expm1(y_test)
y_pred_actual = np.expm1(y_pred)

print("MAE:", metrics.mean_absolute_error(y_test_actual, y_pred_actual))
print("RMSE:", np.sqrt(metrics.mean_squared_error(y_test_actual, y_pred_actual)))

# =========================
# SAVE MODEL + ENCODERS
# =========================

# IMPORTANT (Flask ke liye)
city_mean = df.groupby('city')['rent'].mean()
locality_mean = df.groupby('locality')['rent'].mean()

pickle.dump(model, open(os.path.join(MODEL_PATH, "house_model.pkl"), "wb"))
pickle.dump(city_mean, open(os.path.join(MODEL_PATH, "city_mean.pkl"), "wb"))
pickle.dump(locality_mean, open(os.path.join(MODEL_PATH, "locality_mean.pkl"), "wb"))

print("\n✅ Model and encoders saved successfully!")

NameError: name 'datasets_path' is not defined